In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile requirements-dev.txt
pytest==8.4.2
huggingface_hub

Writing requirements-dev.txt


In [3]:
!pip install -r requirements-dev.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytest]

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [5]:
!mkdir -p tests

In [6]:
%%writefile tests/test_pages.py
"""Day 32：自動測試每一頁。執行：python -m pytest -q"""
import json
from pathlib import Path

import pytest
from streamlit.testing.v1 import AppTest

ROOT = Path(__file__).resolve().parent.parent
PAGES = ["pages/home.py", "pages/academy.py", "pages/watch.py", "pages/predict.py"]

def open_page(page):
    at = AppTest.from_file(str(ROOT / "app.py"), default_timeout=120).run()
    if page != "pages/home.py":
        at.switch_page(page).run()
    return at

@pytest.mark.parametrize("page", PAGES)
def test_page_runs_without_error(page):
    at = open_page(page)
    assert not at.exception, [e.value for e in at.exception]

Writing tests/test_pages.py


In [7]:
!python -m pytest -q

....                                                                     [100%]
4 passed in 1.30s


In [8]:
%%writefile -a tests/test_pages.py
def test_every_stat_card():
    at = open_page("pages/academy.py")
    stats = json.loads((ROOT / "content" / "stats.json").read_text(encoding="utf-8"))
    for s in stats:
        at.radio[0].set_value(s["key"]).run()
        assert not at.exception, s["key"]

def test_lineup_needs_nine_players():
    at = open_page("pages/predict.py")
    ms = at.multiselect[0]
    ms.unselect(ms.value[0]).run()
    assert any("9 位打者" in i.value for i in at.info)

def test_series_probabilities_add_up():
    import pandas as pd
    import sys
    sys.path.insert(0, str(ROOT))
    from src import config, model
    teams = pd.read_csv(ROOT / "data/clean/teams.csv").set_index("team")
    a, b = config.TS_TEAMS
    p, results, _ = model.simulate_series(teams.loc[a, "pythag"], teams.loc[b, "pythag"],
                                          config.TS_HOME_PATTERN)
    assert 0 < p < 1
    assert sum(results.values()) == 10_000

Appending to tests/test_pages.py


In [9]:
%%writefile scripts/check_before_deploy.py
"""上線前檢查：跑頁面測試和實驗 1、2，結果寫進 logs/stability_log.csv；沒通過就回傳錯誤碼，不准部署。"""
import datetime as dt
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent

tests_ok = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT).returncode == 0
subprocess.run([sys.executable, "scripts/run_experiments.py"], cwd=ROOT, check=True)
summary = json.loads((ROOT / "data/clean/stability_summary.json").read_text(encoding="utf-8"))

row = {"時間": dt.datetime.now().strftime("%Y-%m-%d %H:%M"), "頁面測試": tests_ok,
       "實驗1": summary["exp1"]["通過"], "實驗2": summary["exp2"]["通過"],
       "實驗3區間寬度": round(summary["exp3"]["區間寬度"], 3), "實驗4最大差距": round(summary["exp4"]["最大差距"], 3)}
log = ROOT / "logs" / "stability_log.csv"
log.parent.mkdir(exist_ok=True)
new = not log.exists()
with log.open("a", encoding="utf-8") as f:
    if new:
        f.write(",".join(row) + "\n")
    f.write(",".join(str(v) for v in row.values()) + "\n")

ok = tests_ok and row["實驗1"] and row["實驗2"]
print("檢查通過，可以部署" if ok else "檢查沒通過，先不要部署")
sys.exit(0 if ok else 1)

Writing scripts/check_before_deploy.py


In [10]:
!python scripts/check_before_deploy.py
!cat logs/stability_log.csv

.......                                                                  [100%]
7 passed in 1.69s
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
exp1 {"平均": 0.67072, "標準差": 0.004571380233734595, "通過": true}
exp2 {"最大變動": 0.053099999999999925, "先發局數": {"5": 0.5057455641084038, "6": 0.4998950628748998, "7": 0.49373908457796556}, "看好的球隊不變": true, "通過": true}
exp3 {"奪冠機率95%區間": [0.345375, 0.90375], "區間寬度": 0.5583750000000001, "

In [11]:
!python scripts/update_data.py   # 需要更新資料時才跑
!python scripts/deploy.py        # 自動先檢查，通過才上傳

半季戰績來源：中職官網；交叉比對全部一致
球隊 6、打者 183、投手 181；wOBA 倍數 = 1.001、FIP 常數 = 3.02
逐打席 27268 筆；聯盟 wOBA 0.319
Traceback (most recent call last):
  File "/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/scripts/deploy.py", line 9, in <module>
    if subprocess.run([sys.executable, "scripts/check_before_deploy.py"], cwd=ROOT).returncode != 0:
       ^^^^^^^^^^
NameError: name 'subprocess' is not defined. Did you forget to import 'subprocess'?


In [12]:
!python scripts/update_data.py   # 需要更新資料時才跑
!python scripts/deploy.py        # 自動先檢查，通過才上傳

半季戰績來源：中職官網；交叉比對全部一致
球隊 6、打者 183、投手 181；wOBA 倍數 = 1.001、FIP 常數 = 3.02
逐打席 27268 筆；聯盟 wOBA 0.319
.......                                                                  [100%]
7 passed in 1.29s
2026-11-05 20:48:04.989 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:48:04.989 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:48:04.989 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:48:04.989 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:48:04.989 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
exp1 {"平均": 0.67072, "標準差": 0.004571380233734595, "通過": true}
exp2 {"最大變動": 0.053099999999999925, "先發局數": {"5": 0.5057455641084038, "6": 0.4998950628748998, "7": 0.49373908457796556}, "看

In [13]:
!git log -1 --format="%ci %s" prediction-locked
!cat data/clean/prediction_locked.json

2026-10-29 18:51:57 -0700 鎖定台灣大賽預測
{
  "locked_at": "2026-10-29T18:51:57",
  "teams": [
    "味全龍",
    "中信兄弟"
  ],
  "pythag": {
    "味全龍": 0.5784,
    "中信兄弟": 0.5082
  },
  "home_pattern": [
    true,
    true,
    false,
    false,
    true,
    true,
    true
  ],
  "home_advantage": 0.023,
  "game_prob_A_home": 0.5934,
  "game_prob_A_away": 0.5474,
  "champion_prob": {
    "味全龍": 0.6763,
    "中信兄弟": 0.3237
  },
  "result_dist": {
    "4-0": 0.1095,
    "4-1": 0.1918,
    "4-2": 0.2075,
    "4-3": 0.1675,
    "3-4": 0.1132,
    "2-4": 0.1015,
    "1-4": 0.0737,
    "0-4": 0.0353
  },
  "most_likely": "4-2"
}

In [14]:
%%writefile -a src/fetch.py
def find_season_id(year, keyword):
    """在中職所有賽季中，找出標題含有 keyword（例如「台灣大賽」）的賽季代碼；找不到回傳 None。"""
    seasons = get_json("https://www.rebas.tw/api/leagues/CPBL")["data"]["seasons"]
    for s in seasons:
        if s.get("year") == year and keyword in s.get("title", ""):
            return s["uniqid"]
    return None

Appending to src/fetch.py


In [19]:
%%writefile scripts/reveal.py
"""揭曉：抓台灣大賽實際比分，跟鎖定的預測對照，結果存成 data/clean/reveal.json。"""
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
import pandas as pd

from src import config, fetch

YEAR = 2026   # 測試
locked = json.loads((ROOT / "data/clean/prediction_locked.json").read_text(encoding="utf-8"))
A, B = locked["teams"]

season_id = fetch.find_season_id(YEAR, config.POSTSEASON_TITLE)
games = fetch.fetch_games(season_id, refresh=True).sort_values("date").reset_index(drop=True)
games.to_csv(ROOT / "data/clean/postseason.csv", index=False)

rows = []
for i, g in games.iterrows():
    a_home = g["home"] == A                               # 依實際主客場，挑鎖定時算好的勝率
    p_a = locked["game_prob_A_home"] if a_home else locked["game_prob_A_away"]
    winner = g["home"] if g["winner_side"] == "HOME" else g["away"]
    rows.append({"場次": f"G{i + 1}", "日期": g["date"], "主場": g["home"],
                 "比分": f"{g['away']} {g['away_runs']} : {g['home_runs']} {g['home']}",
                 f"{A} 勝率": p_a, "模型看好": A if p_a >= 0.5 else B, "實際勝方": winner,
                 "A 贏": winner == A, "主場贏": g["winner_side"] == "HOME"})
table = pd.DataFrame(rows)
table["猜對"] = table["模型看好"] == table["實際勝方"]

wins_a = int(table["A 贏"].sum())
wins_b = len(table) - wins_a
actual = f"{wins_a}-{wins_b}"
champion = A if wins_a > wins_b else B
reveal = {
    "season_id": season_id,
    "actual_score": actual,
    "actual_champion": champion,
    "predicted_champion": max(locked["champion_prob"], key=locked["champion_prob"].get),
    "prob_of_actual_score": locked["result_dist"].get(actual, 0),
    "games": len(table),
    "hit_rate": float(table["猜對"].mean()),
    "home_wins_rate": float(table["主場贏"].mean()),        # 簡單方法：主場球隊必勝
    "brier": float(((table[f"{A} 勝率"] - table["A 贏"]) ** 2).mean()),
}
(ROOT / "data/clean/reveal.json").write_text(json.dumps(reveal, ensure_ascii=False, indent=2), encoding="utf-8")
table.to_csv(ROOT / "data/clean/reveal_games.csv", index=False)
print(table[["場次", "比分", "模型看好", "實際勝方", "猜對"]].to_string(index=False))
print(json.dumps(reveal, ensure_ascii=False, indent=2))

Overwriting scripts/reveal.py


In [20]:
!python scripts/reveal.py

場次             比分 模型看好 實際勝方    猜對
G1 中信兄弟 0 : 3 味全龍  味全龍  味全龍  True
G2 中信兄弟 1 : 3 味全龍  味全龍  味全龍  True
G3 味全龍 6 : 0 中信兄弟  味全龍  味全龍  True
G4 味全龍 4 : 5 中信兄弟  味全龍 中信兄弟 False
G5 中信兄弟 4 : 3 味全龍  味全龍 中信兄弟 False
G6 中信兄弟 0 : 6 味全龍  味全龍  味全龍  True
{
  "season_id": "CPBL-2026-LU",
  "actual_score": "4-2",
  "actual_champion": "味全龍",
  "predicted_champion": "味全龍",
  "prob_of_actual_score": 0.2075,
  "games": 6,
  "hit_rate": 0.6666666666666666,
  "home_wins_rate": 0.6666666666666666,
  "brier": 0.22543129333333334
}


In [17]:
%%writefile -a src/data.py
@st.cache_data
def load_reveal():
    """最後一天揭曉的結果；還沒揭曉就回傳 (None, None)。"""
    path = CLEAN / "reveal.json"
    if not path.exists():
        return None, None
    return json.loads(path.read_text(encoding="utf-8")), pd.read_csv(CLEAN / "reveal_games.csv")

Appending to src/data.py


In [18]:
%%writefile -a pages/predict.py
reveal, reveal_games = load_reveal()
if reveal:
    st.header("揭曉：預測 vs 實際")
    c1, c2, c3 = st.columns(3)
    with c1:
        ui.kpi("實際結果", f"{reveal['actual_champion']} 奪冠", f"系列賽 {reveal['actual_score']}（{A_NAME} 的勝場在前）")
    with c2:
        ui.kpi("模型預測", f"{reveal['predicted_champion']} 奪冠",
               f"鎖定時給實際比數的機率：{reveal['prob_of_actual_score']:.1%}")
    with c3:
        ui.kpi("單場命中率", f"{reveal['hit_rate']:.0%}",
               f"{reveal['games']} 場；主場球隊勝率 {reveal['home_wins_rate']:.0%}")
    st.dataframe(reveal_games[["場次", "日期", "主場", "比分", "模型看好", "實際勝方", "猜對"]],
                 hide_index=True, width="stretch")

Appending to pages/predict.py
